# Capability: text_extraction

The `text_extraction` capability scans free-form text for field values. It is the default extraction strategy for `STRING` fields — the planner assigns it first when no format hint or regex is specified.

This capability looks for natural-language patterns: named entities, nearby keywords, structural cues like colons and newlines.

In [ ]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic


class PurchaseOrder(BaseModel):
    supplier: str
    order_number: str
    department: str


sample = """
PO-2026-0099
Supplier: ABC Parts Ltd
Department: Engineering
Date: 2026-06-15
"""
result = paxman.normalize(sample, PurchaseOrder)
print(f"Status: {result.status.name}")
print(f"Data:   {result.normalized_data}")
print(f"Unresolved: {result.unresolved_fields}")

In [ ]:
# Inspect evidence for the supplier field
supplier_evidence = result.evidence.get("supplier")
if supplier_evidence:
    for ev in supplier_evidence.evidence:
        print(f"  Value: {ev.value!r}")
        print(f"  Confidence: {ev.confidence.name}")
        print(f"  Source: {ev.evidence_ref}")
        print()

## How it works

1. The **planner** identifies `supplier`, `order_number`, and `department` as `STRING` fields.
2. It assigns `text_extraction` as the first pass (no format hint, no regex).
3. The **executor** runs the capability, collecting evidence for each field.
4. The **reconciler** grades the evidence and assigns final confidence.

> **Reference:** See `src/paxman/capabilities/v1/text_extraction.py` for the implementation.

## Try it yourself

- Add a field the input doesn't contain (e.g. `vat_number`) — it should appear in `unresolved_fields`.
- Change the input format from structured (label: value) to prose ("The supplier is ABC Parts Ltd") and see if text_extraction still resolves it.
- Inspect the `diagnostics` list for warning messages.